# Seq2Seq 모델 Q&A Chatbot 구현

1. QnA 데이터셋을 찾아서 처리해서 준비한다. (전처리 전반)
2. Encoder, Decoder, Seq2Seq(Encoder+Decoder) 모델을 만든다.
3. 1에서 준비한 데이터로 2에서 만든 모델을 학습시킨다.
4. Chatbot을 만든다. (모델 추론 + while문)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/songys/Chatbot_data/refs/heads/master/ChatbotData.csv')
df = df[['Q', 'A']]
df

,Q,A
0,12시 땡!,하루가 또 가네요.
1,1지망 학교 떨어졌어,위로해 드립니다.
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.
4,PPL 심하네,눈살이 찌푸려지죠.
...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.
11820,흑기사 해주는 짝남.,설렜겠어요.
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [6]:
encoder_inputs = []
decoder_inputs = []
decoder_targets = []

for q,a in zip(df['Q'], df['A']): # zip() 함수 : 여러 개의 iterable 객체를 병렬로 순회할 수 있게 해주는 함수
    q = str(q).strip() # strip() 함수 : 문자열의 양쪽 끝에서 공백을 제거하는 함수
    a = str(a).strip()

    

    decoder_input = '<sos> ' + a
    decoder_target = a + ' <eos>'

    encoder_inputs.append(q)
    decoder_inputs.append(decoder_input)
    decoder_targets.append(decoder_target)

In [7]:
print(len(encoder_inputs), len(decoder_inputs), len(decoder_targets))
print(encoder_inputs[5000])
print(decoder_inputs[5000])
print(decoder_targets[5000])

11823 11823 11823
학원폭력 짜증나
<sos> 학교 폭력은 범죄에요.
학교 폭력은 범죄에요. <eos>


In [8]:
BATCH_SIZE = 64
MAX_VOCAB_SIZE = 10000
EMBEDDING_DIM = 100
LATENT_DIM = 512

In [9]:
encoder_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE) # 단어 집합의 최대 크기를 MAX_VOCAB_SIZE로 제한
encoder_tokenizer.fit_on_texts(encoder_inputs) # fit_on_texts() : 텍스트 데이터를 토큰화하여 단어 집합을 구축하는 메서드
encoder_seqs = encoder_tokenizer.texts_to_sequences(encoder_inputs)

encoder_num_words  = len(encoder_tokenizer.word_index) + 1
encoder_max_len = max(len(s) for s in encoder_seqs)

print(f'{encoder_num_words = }')
print(f'{encoder_max_len = }')  

encoder_num_words = 13399
encoder_max_len = 13


In [10]:
decoder_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters='')
decoder_tokenizer.fit_on_texts(decoder_inputs + decoder_targets) 

decoder_input_seqs = decoder_tokenizer.texts_to_sequences(decoder_inputs)

decoder_target_seqs = decoder_tokenizer.texts_to_sequences(decoder_targets)

decoder_num_words = len(decoder_tokenizer.word_index) + 1 

decoder_max_len = max(len(s) for s in decoder_input_seqs)

print(f'{decoder_num_words = }')
print(f'{decoder_max_len = }')

decoder_num_words = 10008
decoder_max_len = 22


In [11]:

encoder_input_data = pad_sequences(encoder_seqs, maxlen=encoder_max_len, padding='pre')
decoder_input_data = pad_sequences(decoder_input_seqs, maxlen=decoder_max_len, padding='post')
decoder_target_data = pad_sequences(decoder_target_seqs, maxlen=decoder_max_len, padding='post')

print(encoder_input_data.shape)
print(decoder_input_data.shape)
print(decoder_target_data.shape)

print(encoder_input_data[1000])
print([encoder_tokenizer.index_word[s] for s in encoder_input_data[1000] if s != 0])

print(decoder_input_data[1000])
print([decoder_tokenizer.index_word[s] for s in decoder_input_data[1000] if s != 0])

print(decoder_target_data[1000])
print([decoder_tokenizer.index_word[s] for s in decoder_target_data[1000] if s != 0])

(11823, 13)
(11823, 22)
(11823, 22)
[   0    0    0    0    0    0    0  646  138    3   74   41 2832]
['노래방', '걸', '거', '같은데', '뭐', '부르지']
[   1 1779 3493    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
['<sos>', '달달한', '노래요.']
[1779 3493    2    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
['달달한', '노래요.', '<eos>']


---

In [12]:
class NMTDataset(Dataset):
  def __init__(self, encoder_inputs, decoder_inputs, decoder_targets):
    super().__init__()
    self.encoder_inputs = encoder_inputs
    self.decoder_inputs = decoder_inputs
    self.decoder_targets = decoder_targets

  def __len__(self):
    return len(self.encoder_inputs)


  def __getitem__(self, index):
    return (
        torch.tensor(self.encoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_inputs[index], dtype=torch.long),
        torch.tensor(self.decoder_targets[index], dtype=torch.long),
    )

In [13]:
train_index, val_index = train_test_split(range(len(encoder_input_data)), random_state=0)
print(len(train_index), len(val_index))

train_dataset = NMTDataset(
    encoder_input_data[train_index],
    decoder_input_data[train_index],
    decoder_target_data[train_index]
)
val_dataset = NMTDataset(
    encoder_input_data[val_index],
    decoder_input_data[val_index],
    decoder_target_data[val_index]
)

8867 2956


In [14]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

#### 모델 만들기

In [15]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim, embedding_matrix=None):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    if embedding_matrix is not None:
      self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)


  # 문장의 의미를 hidden state와 cell state에 담아 Decoder로 넘기는 역할
  def forward(self, X):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X)
    return h_s, c_s

In [16]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, embedding_dim, latent_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.lstm = nn.LSTM(embedding_dim, latent_dim, batch_first=True)
    self.fc = nn.Linear(latent_dim, vocab_size)

  def forward(self, X, hidden, cell):
    X = self.embedding(X)
    output, (h_s, c_s) = self.lstm(X, (hidden, cell))
    logits = self.fc(output)
    return logits, h_s, c_s

In [17]:
class Seq2Seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, source, target):
    h_s, c_s = self.encoder(source)
    output, h_s, c_s = self.decoder(target, h_s, c_s)
    return output

In [18]:
encoder = Encoder(encoder_num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(decoder_num_words, EMBEDDING_DIM, LATENT_DIM)

model = Seq2Seq(encoder, decoder)

#### 모델 학습

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

encoder = Encoder(encoder_num_words, EMBEDDING_DIM, LATENT_DIM)
decoder = Decoder(decoder_num_words, EMBEDDING_DIM, LATENT_DIM)
model = Seq2Seq(encoder, decoder).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

epochs = 100

train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(epochs):
    model.train()
    train_loss, train_correct, train_tokens = 0, 0, 0

    for enc_inputs, dec_inputs, dec_targets in train_loader:
        enc_inputs = enc_inputs.to(device)
        dec_inputs = dec_inputs.to(device)
        dec_targets = dec_targets.to(device)

        optimizer.zero_grad()

        output = model(enc_inputs, dec_inputs)
        output = output.view(-1, output.size(-1))
        dec_targets = dec_targets.view(-1)

        loss = criterion(output, dec_targets)
        loss.backward()
        optimizer.step()

        preds = output.argmax(dim=-1)
        train_loss += loss.item()

        mask = dec_targets != 0
        correct = (preds == dec_targets) & mask
        train_correct += correct.sum().item()
        train_tokens += mask.sum().item()

    train_loss /= len(train_loader)
    train_acc = train_correct / train_tokens
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    model.eval()
    with torch.no_grad():
        val_loss, val_correct, val_tokens = 0, 0, 0

        for enc_inputs, dec_inputs, dec_targets in val_loader:
            enc_inputs = enc_inputs.to(device)
            dec_inputs = dec_inputs.to(device)
            dec_targets = dec_targets.to(device)

            output = model(enc_inputs, dec_inputs)
            output = output.view(-1, output.size(-1))
            dec_targets = dec_targets.view(-1)

            loss = criterion(output, dec_targets)

            preds = output.argmax(dim=-1)
            val_loss += loss.item()

            mask = dec_targets != 0
            correct = (preds == dec_targets) & mask
            val_correct += correct.sum().item()
            val_tokens += mask.sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_tokens
        val_losses.append(val_loss)
        val_accs.append(val_acc)

    print(f'Epoch {epoch+1}/{epochs} TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')

Epoch 1/100 TrainLoss=7.0137 TrainAcc=0.2118 ValLoss=6.6224 ValAcc=0.2241
Epoch 2/100 TrainLoss=6.2158 TrainAcc=0.2303 ValLoss=6.4786 ValAcc=0.2375
Epoch 3/100 TrainLoss=5.6532 TrainAcc=0.2468 ValLoss=6.1169 ValAcc=0.2551
Epoch 4/100 TrainLoss=4.6829 TrainAcc=0.2996 ValLoss=5.7644 ValAcc=0.2928
Epoch 5/100 TrainLoss=3.5147 TrainAcc=0.4090 ValLoss=5.4591 ValAcc=0.3361
Epoch 6/100 TrainLoss=2.4087 TrainAcc=0.5646 ValLoss=5.3772 ValAcc=0.3673
Epoch 7/100 TrainLoss=1.4953 TrainAcc=0.7406 ValLoss=5.3328 ValAcc=0.3931
Epoch 8/100 TrainLoss=0.8552 TrainAcc=0.8698 ValLoss=5.4008 ValAcc=0.4034
Epoch 9/100 TrainLoss=0.4637 TrainAcc=0.9449 ValLoss=5.4369 ValAcc=0.4069
Epoch 10/100 TrainLoss=0.2540 TrainAcc=0.9770 ValLoss=5.4986 ValAcc=0.4075
Epoch 11/100 TrainLoss=0.1573 TrainAcc=0.9858 ValLoss=5.5449 ValAcc=0.4072
Epoch 12/100 TrainLoss=0.1147 TrainAcc=0.9886 ValLoss=5.5826 ValAcc=0.4061
Epoch 13/100 TrainLoss=0.0917 TrainAcc=0.9895 ValLoss=5.6530 ValAcc=0.4068
Epoch 14/100 TrainLoss=0.0779 Trai

#### chatbot(모델 학습 + while 문)

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def translate(input_seq, model, decoder_tokenizer, max_len=decoder_max_len, device=device):
  model = model.to(device)
  model.eval()
  encoder = model.encoder
  decoder = model.decoder

  input_seq = torch.tensor(input_seq, dtype=torch.long).to(device)

  # Encoder 처리
  with torch.no_grad():
    hidden, cell = encoder(input_seq)

  # Decoder 출력(Auto Regressive)
  sos_index = decoder_tokenizer.word_index['<sos>']
  eos_index = decoder_tokenizer.word_index['<eos>']

  output_sentences = []

  target_seq = torch.tensor([[sos_index]], dtype=torch.long).to(device)

  with torch.no_grad():
    for _ in range(max_len):
      output, hidden, cell = decoder(target_seq, hidden, cell)  # (batch_size, seq_len, vocab_size)
      proba = output.squeeze(1).softmax(dim=-1)                 # (batch_size, vocab_size)
      pred = proba.argmax(dim=-1).detach().cpu().item()

      if pred == eos_index:
        break

      if pred > 0:
        word = decoder_tokenizer.index_word[pred]
        output_sentences.append(word)

      target_seq = torch.tensor([[pred]], dtype=torch.long).to(device)

  return ' '.join(output_sentences)

In [29]:
while True:
    input_ = input('영어 원문 대신 질문 입력 (종료하려면 quit): ')

    if input_.lower() == 'quit':
        print('종료합니다.')
        break

    input_seq = encoder_tokenizer.texts_to_sequences([input_])
    input_seq = pad_sequences(input_seq, maxlen=encoder_max_len, padding='pre')

    output = translate(input_seq, model, decoder_tokenizer)

    print(f'입력 문장: {input_}')
    print(f'모델 추론 문장: {output}')
    print()

입력 문장: 취업하고 싶어
모델 추론 문장: 쉽지 않을 거예요.

입력 문장: 너무하네 진짜로
모델 추론 문장: 자신에게는 너그러웠으면 합니다.

입력 문장: 배고파
모델 추론 문장: 얼른 맛난 음식 드세요.

입력 문장: 집에 가고 싶어
모델 추론 문장: 집이 최고죠.

입력 문장: 놀러 가고 싶어
모델 추론 문장: 저도 같이 가요.

입력 문장: 어어 오지마라
모델 추론 문장: 좋은 이별은 없다고 생각해요.

종료합니다.
